<a href="https://colab.research.google.com/github/wahyuherdiansyah32-maker/tugas-steganografi/blob/main/GANJIL_Wahyu.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import files
uploaded = files.upload()

Saving Ganjil.csv to Ganjil.csv


In [2]:
import pandas as pd

df = pd.read_csv('Ganjil.csv', sep=';')  # WAJIB pakai ;
print(df.columns)
df.head()

Index(['No', 'Berita'], dtype='object')


,No,Berita
0,1.0,"JAKARTA, KOMPAS.com - Komisi Pemberantasan Kor..."
1,NaN,NaN
2,NaN,Sumber: https://nasional.kompas.com/read/2026/...
3,NaN,NaN
4,NaN,NaN


In [3]:
import re
import string

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'http\S+', '', text)
    text = text.translate(str.maketrans('', '', string.punctuation))
    text = text.strip()
    return text

df['cleaning'] = df['Berita'].apply(clean_text)

In [4]:
df['tokens'] = df['cleaning'].apply(lambda x: x.split())

In [5]:
normal_dict = {
    "gk": "tidak",
    "ga": "tidak",
    "yg": "yang",
    "dr": "dari",
    "dlm": "dalam",
    "krn": "karena"
}

def normalize(tokens):
    return [normal_dict.get(word, word) for word in tokens]

df['normalized'] = df['tokens'].apply(normalize)

In [6]:
from nltk.corpus import stopwords
import nltk

nltk.download('stopwords')
stop_words = set(stopwords.words('indonesian'))

df['no_stopwords'] = df['normalized'].apply(
    lambda tokens: [w for w in tokens if w not in stop_words]
)

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [7]:
from nltk.stem import WordNetLemmatizer
import nltk

nltk.download('wordnet')

lemmatizer = WordNetLemmatizer()

df['lemmatization'] = df['no_stopwords'].apply(
    lambda tokens: [lemmatizer.lemmatize(word) for word in tokens]
)

[nltk_data] Downloading package wordnet to /root/nltk_data...


In [8]:
!pip install Sastrawi

from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
factory = StemmerFactory()
stemmer = factory.create_stemmer()

df['stemming'] = df['no_stopwords'].apply(
    lambda tokens: [stemmer.stem(word) for word in tokens]
)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 209.7/209.7 kB 13.8 MB/s eta 0:00:00


In [11]:
lemmatizer = WordNetLemmatizer()

def lemmatize(tokens):
    return [lemmatizer.lemmatize(word) for word in tokens]

df['lemmatization'] = df['no_stopwords'].apply(lemmatize)

In [9]:
# hapus string kosong
df = df.replace(r'^\s*$', pd.NA, regex=True)

# hapus baris kosong total
df = df.dropna(how='all')

# hapus jika kolom utama kosong
df = df.dropna(subset=['Berita'])

# hapus token kosong
df = df[df['no_stopwords'].apply(lambda x: len(x) > 0)]

# reset index
df = df.reset_index(drop=True)

In [10]:
df.to_csv('hasil_final_bersih.csv', index=False)

from google.colab import files
files.download('hasil_final_bersih.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>